> Organized from the original notebook. Saved outputs are historical and have not been regenerated. See `docs/analysis.md` before interpreting results.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "project_paths.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Launch Jupyter inside the Cancer-overview-ml project.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from project_paths import *
ensure_directories()


In [7]:
import os
import glob
import pandas as pd
from collections import Counter


In [8]:
cleaned_dir = str(CLEANED_TEXT_DIR)
output_dir = str(NGRAM_DIR)
os.makedirs(output_dir, exist_ok=True)

In [9]:
total_unigram_counts = Counter()
document_unigram_counts = Counter()

total_bigram_counts = Counter()
document_bigram_counts = Counter()

total_trigram_counts = Counter()
document_trigram_counts = Counter()

total_quadgram_counts = Counter()
document_quadgram_counts = Counter()

file_list = glob.glob(os.path.join(cleaned_dir, "*.txt"))

In [10]:
for filepath in file_list:
    with open(filepath, 'r', encoding='utf-8') as f:
        tokens = f.read().split()
        
       
        total_unigram_counts.update(tokens)
        for token in set(tokens):
            document_unigram_counts[token] += 1
      
        bigrams = list(zip(tokens, tokens[1:]))
        total_bigram_counts.update(bigrams)
        for bigram in set(bigrams):
            document_bigram_counts[bigram] += 1
        
        
        trigrams = list(zip(tokens, tokens[1:], tokens[2:]))
        total_trigram_counts.update(trigrams)
        for trigram in set(trigrams):
            document_trigram_counts[trigram] += 1
        
       
        quadgrams = list(zip(tokens, tokens[1:], tokens[2:], tokens[3:]))
        total_quadgram_counts.update(quadgrams)
        for quadgram in set(quadgrams):
            document_quadgram_counts[quadgram] += 1

In [11]:
def ngram_df(total_counts, doc_counts, n, filename):
    if n == 1:
        gram_list = list(total_counts.keys())
        gram_str = gram_list
    else:
        gram_list = list(total_counts.keys())
        gram_str = [' '.join(gram) for gram in gram_list]
    df = pd.DataFrame({
        f'{n}-gram': gram_str,
        'documents': [doc_counts[gram] for gram in gram_list],
        'total freq': [total_counts[gram] for gram in gram_list],
        'avg': [total_counts[gram]/doc_counts[gram] for gram in gram_list]
    })
    df = df.sort_values(by='total freq', ascending=False)
    df.to_csv(os.path.join(output_dir, filename), index=False)

In [12]:
ngram_df(total_unigram_counts, document_unigram_counts, 1, "unigram_sorted.csv")
ngram_df(total_bigram_counts, document_bigram_counts, 2, "bigram_sorted.csv")
ngram_df(total_trigram_counts, document_trigram_counts, 3, "trigram_sorted.csv")
ngram_df(total_quadgram_counts, document_quadgram_counts, 4, "quadgram_sorted.csv")